# BirdCLEF 2026 Training v25 — Larger PerchGRU + Mel Models
## Upgraded BiGRU (hidden=768, layers=3, attention head) + ResNet18 & EfficientNet-B0 on correct species list

### Changes vs v23
- **GRU**: hidden 512→768, layers 2→3, added attention-pooling head for better sequence aggregation
- **Mel branch**: ResNet18 + EfficientNet-B0 trained on same taxonomy.csv as GRU (fixes species-list mismatch)
- **Joint training file**: both model types trained in one notebook
- **More epochs**: 30 (was 25), patience 8 (was 7)

### Required Kaggle inputs
1. `birdclef-2026` (competition data)
2. `birdclef-2026-perch-embs-v3` (Perch embeddings, from precompute-kaggle-v3)

### Outputs
- `perch_gru_v25_fold*.pt` — upgraded GRU checkpoints
- `resnet18_v25_fold*.pt` — ResNet18 mel checkpoints
- `efficientnet_b0_v25_fold*.pt` — EfficientNet-B0 mel checkpoints
→ Output tab → New Dataset → name: `birdclef-2026-weights-v25`

In [ ]:
# === CELL 1: IMPORTS & CONFIG ===
import os, json, ast, copy, random, subprocess, sys
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import librosa
import soundfile as sf

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR
from torch.cuda.amp import autocast, GradScaler
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from tqdm import tqdm
import timm

# ---- GPU compatibility check ----
# PyTorch >=2.0 dropped sm_60 (Tesla P100). Fall back to CPU if detected.
def _resolve_device():
    if not torch.cuda.is_available():
        print('No CUDA device found — using CPU.')
        return 'cpu'
    major, minor = torch.cuda.get_device_capability(0)
    name = torch.cuda.get_device_name(0)
    sm = major * 10 + minor
    min_sm = 70  # PyTorch >=2.0 minimum
    if sm < min_sm:
        print(f'WARNING: {name} is sm_{sm} — below PyTorch minimum sm_{min_sm}.')
        print('Attempting to install a compatible PyTorch (this takes ~2 min)...')
        _pkg = (
            'torch==1.13.1+cu116 torchvision==0.14.1+cu116 '
            '--extra-index-url https://download.pytorch.org/whl/cu116'
        )
        result = subprocess.run(
            [sys.executable, '-m', 'pip', 'install', '-q'] + _pkg.split(),
            capture_output=True, text=True
        )
        if result.returncode == 0:
            print('Compatible PyTorch installed — please RESTART the kernel and re-run.')
        else:
            print('Install failed. Falling back to CPU (slow but functional).')
            print(result.stderr[-500:])
        return 'cpu'
    print(f'GPU OK: {name} (sm_{sm})')
    return 'cuda'

CFG = dict(
    # Training
    epochs         = 30,
    warmup_epochs  = 4,
    lr             = 5e-4,
    batch_size     = 64,
    patience       = 8,
    num_workers    = 2,
    seed           = 42,
    swa_start_frac = 0.6,
    secondary_label_weight   = 0.3,
    soundscape_sample_weight = 0.8,
    checkpoint_tag = 'v25',
    folds          = 5,
    device         = _resolve_device(),

    # GRU branch (upgraded)
    perch_emb_dim  = 1536,
    perch_emb_noise= 0.02,
    gru_hidden     = 768,   # was 512
    gru_layers     = 3,     # was 2
    gru_dropout    = 0.3,
    max_seq_len    = 60,

    # Mel branch
    mel_sr         = 16000,
    mel_seconds    = 5,
    n_mels         = 64,
    n_fft          = 1024,
    hop_length     = 320,
    fmin           = 60,
    fmax           = 8000,
    mel_batch      = 32,
    mel_lr         = 3e-4,
    mel_epochs     = 25,
    mel_patience   = 7,
)
CFG['mel_target'] = CFG['mel_sr'] * CFG['mel_seconds']  # 80,000

random.seed(CFG['seed'])
np.random.seed(CFG['seed'])
torch.manual_seed(CFG['seed'])
device = torch.device(CFG['device'])
_use_amp = (device.type == 'cuda')

print(f'v25 — Larger PerchGRU + Mel Models')
print(f'   Device  : {device}  (AMP={_use_amp})')
print(f'   Folds   : {CFG["folds"]}  GRU epochs: {CFG["epochs"]}  Mel epochs: {CFG["mel_epochs"]}')
print(f'   GRU     : hidden={CFG["gru_hidden"]}, layers={CFG["gru_layers"]}, bidirectional=True, attention=True')


In [ ]:
# === CELL 2: PATHS & SPECIES ===
COMP_DIR  = '/kaggle/input/birdclef-2026'
_out_root = '/kaggle/working'

TRAIN_META_CSV  = f'{COMP_DIR}/train.csv'
TAXONOMY_CSV    = f'{COMP_DIR}/taxonomy.csv'
SOUNDSCAPE_ANNO = f'{COMP_DIR}/train_soundscapes_labels.csv'
TRAIN_AUDIO_DIR = f'{COMP_DIR}/train_audio'

# Perch embeddings (for GRU branch)
_embs_candidates = [
    '/kaggle/input/birdclef-2026-perch-embs-v3/perch_embeddings_v3',
    '/kaggle/input/birdclef-2026-perch-embs-v3',
]
EMBD_DIR = next((p for p in _embs_candidates if Path(p).is_dir() and
                 list(Path(p).glob('*.npy'))), None)
if EMBD_DIR is None:
    raise RuntimeError('No embedding .npy files found. Run precompute-kaggle-v3.ipynb first.')
EMBD_DIR = Path(EMBD_DIR)

_all_emb_files = list(EMBD_DIR.glob('*.npy'))
_focal_stems   = {f.stem for f in _all_emb_files if not f.stem.startswith('soundscape_')}
_sc_stems      = {f.stem for f in _all_emb_files if f.stem.startswith('soundscape_')}

if len(_focal_stems) < 100:
    raise RuntimeError(f'Focal embeddings missing in {EMBD_DIR}.')

taxonomy_df = pd.read_csv(TAXONOMY_CSV)
species     = taxonomy_df['primary_label'].astype(str).tolist()
species_set = set(species)
sp_idx      = {lab: i for i, lab in enumerate(species)}
n_classes   = len(species)
df          = pd.read_csv(TRAIN_META_CSV)

with open(f'{_out_root}/species_v25.json', 'w') as f:
    json.dump(species, f)

print(f'EMBD_DIR          : {EMBD_DIR}')
print(f'  Focal embeds    : {len(_focal_stems)}')
print(f'  Soundscape embs : {len(_sc_stems)}')
print(f'  Species         : {n_classes}')
print(f'  Train audio     : {TRAIN_AUDIO_DIR}')

In [ ]:
# === CELL 3: LABEL & SPECTROGRAM HELPERS ===
def parse_secondary(s):
    if pd.isna(s): return []
    t = str(s).strip()
    if t in ('', '[]'): return []
    try:
        lst = ast.literal_eval(t)
        return [str(v) for v in lst] if isinstance(lst, list) else []
    except Exception:
        return []

def row_to_multihot(primary_id: str, secondary_str: str) -> np.ndarray:
    y = np.zeros(n_classes, dtype='float32')
    if str(primary_id) in sp_idx:
        y[sp_idx[str(primary_id)]] = 1.0
    for sid in parse_secondary(secondary_str):
        if sid in sp_idx:
            y[sp_idx[sid]] = CFG['secondary_label_weight']
    return y

def soundscape_to_multihot(label_str: str) -> np.ndarray:
    y = np.zeros(n_classes, dtype='float32')
    for sp in str(label_str).split(';'):
        sp = sp.strip()
        if sp in sp_idx:
            y[sp_idx[sp]] = 1.0
    return y

def _parse_hms(s: str) -> int:
    p = str(s).strip().split(':')
    return int(p[0]) * 3600 + int(p[1]) * 60 + int(p[2])

# Mel filter bank (shared)
_mel_filter = librosa.filters.mel(
    sr=CFG['mel_sr'], n_fft=CFG['n_fft'],
    n_mels=CFG['n_mels'], fmin=CFG['fmin'], fmax=CFG['fmax'],
)

def logmel_from_wave(wave_16k: np.ndarray) -> np.ndarray:
    tgt = CFG['mel_target']
    if len(wave_16k) < tgt:
        wave_16k = np.pad(wave_16k, (0, tgt - len(wave_16k)))
    elif len(wave_16k) > tgt:
        # Random crop during training
        start = random.randint(0, len(wave_16k) - tgt)
        wave_16k = wave_16k[start:start + tgt]
    stft   = librosa.stft(wave_16k, n_fft=CFG['n_fft'], hop_length=CFG['hop_length'],
                          window='hann', center=True)
    power  = np.abs(stft) ** 2
    mel    = _mel_filter @ power
    logmel = np.log1p(mel).astype(np.float32)
    return logmel

print('\u2705 Label & spectrogram helpers defined')

In [ ]:
# === CELL 4: MODEL DEFINITIONS ===

# ---- Attention pooling helper ----
class AttentionPool(nn.Module):
    """Soft-attention over the time dimension: (B, T, D) -> (B, D)"""
    def __init__(self, d_in: int):
        super().__init__()
        self.q = nn.Linear(d_in, 1)
    def forward(self, h: torch.Tensor, mask: torch.Tensor = None):
        # h: (B, T, D)  mask: (B, T) bool
        scores = self.q(h).squeeze(-1)          # (B, T)
        if mask is not None:
            scores = scores.masked_fill(~mask, float('-inf'))
        w = torch.softmax(scores, dim=-1)        # (B, T)
        return (h * w.unsqueeze(-1)).sum(1)       # (B, D)


# ---- PerchGRU v25 (upgraded) ----
class PerchGRU(nn.Module):
    """
    Bidirectional GRU over per-window Perch 1536-d embeddings.
    v25 upgrades: hidden=768, layers=3, attention-pool head for soundscape scoring.
    Input : (B, T, 1536)
    Output: (B, T, n_classes)  per-window logits
    """
    def __init__(self, n_classes: int, emb_dim: int = 1536,
                 hidden: int = 768, n_layers: int = 3, dropout: float = 0.3):
        super().__init__()
        self.proj = nn.Sequential(
            nn.LayerNorm(emb_dim),
            nn.Linear(emb_dim, 768),
            nn.GELU(),
        )
        self.gru = nn.GRU(
            input_size=768, hidden_size=hidden,
            num_layers=n_layers, batch_first=True,
            bidirectional=True,
            dropout=dropout if n_layers > 1 else 0.0,
        )
        d_gru = hidden * 2
        self.attn = AttentionPool(d_gru)
        self.head = nn.Sequential(
            nn.LayerNorm(d_gru),
            nn.Dropout(0.2),
            nn.Linear(d_gru, n_classes),
        )

    def forward(self, x: torch.Tensor, mask: torch.Tensor = None) -> torch.Tensor:
        single = (x.dim() == 2)
        if single:
            x = x.unsqueeze(1)
        z    = self.proj(x)           # (B, T, 768)
        h, _ = self.gru(z)            # (B, T, hidden*2)
        out  = self.head(h)           # (B, T, n_classes)
        if single:
            out = out.squeeze(1)
        return out


# ---- BirdCLEFModel (mel branch) ----
class BirdCLEFModel(nn.Module):
    """ResNet18 or EfficientNet-B0 log-mel classifier."""
    def __init__(self, arch: str, n_classes: int, pretrained: bool = True):
        super().__init__()
        if arch == 'resnet18':
            base    = timm.create_model('resnet18', pretrained=pretrained, in_chans=1)
            n_feats = base.fc.in_features
            base.fc = nn.Identity()
        elif arch == 'efficientnet_b0':
            base            = timm.create_model('efficientnet_b0', pretrained=pretrained, in_chans=1)
            n_feats         = base.classifier.in_features
            base.classifier = nn.Identity()
        else:
            raise ValueError(f'Unknown arch: {arch}')
        self.backbone = base
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.head = nn.Sequential(
            nn.Linear(n_feats, 512),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(512, n_classes),
        )

    def forward(self, x):   # x: (B, 1, n_mels, T_frames)
        feats = self.backbone(x)
        if feats.dim() == 4:
            feats = self.pool(feats).flatten(1)
        return self.head(feats)


_g = PerchGRU(10, 1536, CFG['gru_hidden'], CFG['gru_layers'])
print(f'PerchGRU v25   : {sum(p.numel() for p in _g.parameters())/1e6:.2f}M params')
del _g
_r = BirdCLEFModel('resnet18', 10)
print(f'ResNet18       : {sum(p.numel() for p in _r.parameters())/1e6:.2f}M params')
del _r
_e = BirdCLEFModel('efficientnet_b0', 10)
print(f'EfficientNet-B0: {sum(p.numel() for p in _e.parameters())/1e6:.2f}M params')
del _e
print('\u2705 All model classes defined')

In [ ]:
# === CELL 5: GRU DATASETS ===

class FocalDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, emb_root: Path, train: bool):
        self.df       = frame.reset_index(drop=True)
        self.emb_root = emb_root
        self.train    = train

    def __len__(self): return len(self.df)

    def __getitem__(self, i):
        r   = self.df.iloc[i]
        emb = np.load(self.emb_root / (str(r['filename']) + '.npy')).astype('float32')
        if self.train and random.random() < 0.5:
            emb += np.random.randn(*emb.shape).astype('float32') * CFG['perch_emb_noise']
        x = torch.from_numpy(emb).unsqueeze(0)   # (1, 1536)
        y = torch.from_numpy(
            row_to_multihot(r['primary_label'], r.get('secondary_labels', '[]'))
        ).unsqueeze(0).float()                    # (1, n_classes)
        w = torch.tensor(float(r.get('sample_weight', 1.0)))
        return x, y, w


class SoundscapeSeqDataset(Dataset):
    def __init__(self, seq_groups: list, emb_root: Path, train: bool):
        self.groups   = seq_groups
        self.emb_root = emb_root
        self.train    = train

    def __len__(self): return len(self.groups)

    def __getitem__(self, i):
        grp     = self.groups[i]
        windows = sorted(grp['windows'], key=lambda w: w[1])[:CFG['max_seq_len']]
        T       = len(windows)
        embs    = np.zeros((T, CFG['perch_emb_dim']), dtype='float32')
        labels  = np.zeros((T, n_classes), dtype='float32')
        for t, (stem, end_secs, lv) in enumerate(windows):
            ep = self.emb_root / (stem + '.npy')
            if ep.exists():
                e = np.load(str(ep)).astype('float32')
                if self.train and random.random() < 0.5:
                    e += np.random.randn(*e.shape).astype('float32') * CFG['perch_emb_noise']
                embs[t] = e
            labels[t] = lv
        x = torch.from_numpy(embs)
        y = torch.from_numpy(labels)
        w = torch.tensor(grp.get('weight', 1.0), dtype=torch.float32)
        return x, y, w


def seq_collate(batch):
    xs, ys, ws = zip(*batch)
    max_T  = max(x.shape[0] for x in xs)
    B      = len(xs)
    x_pad  = torch.zeros(B, max_T, CFG['perch_emb_dim'])
    y_pad  = torch.zeros(B, max_T, n_classes)
    mask   = torch.zeros(B, max_T, dtype=torch.bool)
    for i, (x, y) in enumerate(zip(xs, ys)):
        T = x.shape[0]
        x_pad[i, :T] = x
        y_pad[i, :T] = y
        mask[i, :T]  = True
    return x_pad, y_pad, mask, torch.stack(ws)


print('\u2705 GRU datasets defined')

In [ ]:
# === CELL 6: MEL DATASET ===

class MelFocalDataset(Dataset):
    """
    Loads raw audio, computes log-mel on the fly.
    Uses same train.csv / taxonomy.csv as GRU — correct species list.
    """
    def __init__(self, frame: pd.DataFrame, audio_root: str, train: bool):
        self.df         = frame.reset_index(drop=True)
        self.audio_root = Path(audio_root)
        self.train      = train

    def __len__(self): return len(self.df)

    def _load_wave(self, filepath: str) -> np.ndarray:
        try:
            y, sr = sf.read(filepath, always_2d=False)
            if y.ndim == 2: y = y.mean(1)
            if sr != CFG['mel_sr']:
                y = librosa.resample(y.astype(np.float32), orig_sr=sr, target_sr=CFG['mel_sr'])
            return y.astype(np.float32)
        except Exception:
            return np.zeros(CFG['mel_target'], dtype=np.float32)

    def __getitem__(self, i):
        r        = self.df.iloc[i]
        # filename in train.csv is e.g. "amecro/XC12345.ogg"
        filepath = self.audio_root / str(r['filename'])
        wave     = self._load_wave(str(filepath))

        # SpecAugment: time masking during training
        lm = logmel_from_wave(wave)
        if self.train:
            # Frequency masking
            if random.random() < 0.5:
                f0 = random.randint(0, CFG['n_mels'] - 1)
                fw = random.randint(1, min(10, CFG['n_mels'] - f0))
                lm[f0:f0+fw, :] = 0.0
            # Time masking
            if random.random() < 0.5:
                T = lm.shape[1]
                t0 = random.randint(0, max(0, T - 1))
                tw = random.randint(1, min(20, T - t0))
                lm[:, t0:t0+tw] = 0.0

        # Normalize per-clip
        lm = (lm - lm.mean()) / (lm.std() + 1e-6)
        x  = torch.from_numpy(lm).float().unsqueeze(0)  # (1, n_mels, T_frames)
        y  = torch.from_numpy(
            row_to_multihot(r['primary_label'], r.get('secondary_labels', '[]'))
        ).float()
        return x, y


# Quick dataset check
_mel_df = df.copy()
_mel_df['secondary_labels'] = _mel_df['secondary_labels'].fillna('[]') if 'secondary_labels' in _mel_df.columns else '[]'
_mel_df = _mel_df[_mel_df['primary_label'].isin(species_set)].reset_index(drop=True)
print(f'Mel training clips available: {len(_mel_df)}')
print('\u2705 MelFocalDataset defined')

In [ ]:
# === CELL 7: BUILD SEQUENCE GROUPS (for GRU soundscape training) ===
sc_anno = pd.read_csv(SOUNDSCAPE_ANNO)
_sc_label_map = {}
for _, row in sc_anno.iterrows():
    sc_stem  = Path(str(row['filename'])).stem
    end_secs = _parse_hms(row['end'])
    lv       = soundscape_to_multihot(row['primary_label'])
    _sc_label_map[(sc_stem, end_secs)] = lv

_sc_groups_dict = defaultdict(list)
missing_labels  = 0
for f in EMBD_DIR.glob('soundscape_*.npy'):
    stem_part, end_part = f.stem.rsplit('_', 1)
    if not end_part.endswith('s'):
        continue
    end_secs = int(end_part[:-1])
    sc_stem  = stem_part[len('soundscape_'):]
    lv       = _sc_label_map.get((sc_stem, end_secs))
    if lv is None:
        missing_labels += 1
        continue
    _sc_groups_dict[sc_stem].append((f.stem, end_secs, lv))

seq_groups = [
    {'stem': sc_stem, 'windows': windows, 'weight': CFG['soundscape_sample_weight']}
    for sc_stem, windows in _sc_groups_dict.items() if windows
]

focal_df = df.copy()
focal_df['filename'] = focal_df['filename'].apply(lambda x: x.replace('/', '_'))
if 'secondary_labels' not in focal_df.columns:
    focal_df['secondary_labels'] = '[]'
else:
    focal_df['secondary_labels'] = focal_df['secondary_labels'].fillna('[]')
focal_df['sample_weight'] = 1.0
focal_df = focal_df[focal_df['filename'].isin(_focal_stems)].reset_index(drop=True)

print(f'Focal clips (GRU)     : {len(focal_df)}')
print(f'Soundscape sequences  : {len(seq_groups)}')
print(f'  (missing labels     : {missing_labels})')

In [ ]:
# === CELL 7b: PRE-COMPUTE 16kHz WAVEFORM CACHE (run once, ~10-20 min) ===
# Caches resampled 16kHz waveforms so mel training does np.load instead of
# sf.read + librosa.resample every step — speeds up mel training ~50-100x.
import multiprocessing as _mp
import functools as _functools

WAV_CACHE_DIR.mkdir(parents=True, exist_ok=True)

mel_df_for_cache = df[df['primary_label'].isin(species_set)].copy()
_unique_files = mel_df_for_cache['filename'].drop_duplicates().tolist()

def _cache_key(filename):
    return str(filename).replace('/', '_').rsplit('.', 1)[0]

# All args passed explicitly — no global captures, safe for multiprocessing
def _precompute_one(args):
    filename, audio_root, cache_dir, mel_sr, mel_target = args
    key     = str(filename).replace('/', '_').rsplit('.', 1)[0]
    out_npy = Path(cache_dir) / f'{key}.npy'
    if out_npy.exists():
        return 'skip'
    import soundfile as _sf
    import librosa as _lb
    import numpy as _np
    filepath = Path(audio_root) / str(filename)
    try:
        y, sr = _sf.read(str(filepath), always_2d=False)
        if y.ndim == 2: y = y.mean(1)
        if sr != mel_sr:
            y = _lb.resample(y.astype(_np.float32), orig_sr=sr, target_sr=mel_sr)
        _np.save(str(out_npy), y.astype(_np.float32))
        return 'ok'
    except Exception as e:
        return f'err:{e}'

_already = sum(1 for fn in _unique_files if (WAV_CACHE_DIR / (_cache_key(fn) + '.npy')).exists())
_to_do   = len(_unique_files) - _already
print(f'Waveform cache: {_already}/{len(_unique_files)} already done, {_to_do} to compute...')

# Estimate disk usage
_est_gb = _to_do * CFG['mel_target'] * 4 / 1e9
print(f'Est. new disk usage: ~{_est_gb:.1f} GB  (total ~{len(_unique_files) * CFG["mel_target"] * 4 / 1e9:.1f} GB)')

if _to_do > 0:
    _args_list = [
        (fn, TRAIN_AUDIO_DIR, str(WAV_CACHE_DIR), CFG['mel_sr'], CFG['mel_target'])
        for fn in _unique_files
        if not (WAV_CACHE_DIR / (_cache_key(fn) + '.npy')).exists()
    ]
    _n_workers = min(os.cpu_count() or 4, 8)
    with _mp.Pool(_n_workers) as pool:
        results = list(tqdm(
            pool.imap(_precompute_one, _args_list, chunksize=64),
            total=len(_args_list), desc='Caching waveforms'
        ))
    _errors = [r for r in results if r.startswith('err')]
    if _errors:
        print(f'  ⚠️  {len(_errors)} errors (first 5): {_errors[:5]}')

_cached_count = len(list(WAV_CACHE_DIR.glob('*.npy')))
print(f'✅ Cache complete: {_cached_count} files in {WAV_CACHE_DIR}')


In [ ]:
# === CELL 8: TRAIN GRU FOLDS (v25 — hidden=768, layers=3) ===
print('=' * 65)
print(f'v25 PerchGRU Training   {CFG["folds"]} folds  hidden={CFG["gru_hidden"]}  layers={CFG["gru_layers"]}')
print('=' * 65)

_use_amp   = (device.type == 'cuda')
_criterion = nn.BCEWithLogitsLoss(reduction='none')
skf        = StratifiedKFold(n_splits=CFG['folds'], shuffle=True, random_state=CFG['seed'])

gru_fold_scores = []

def _rare_label(primary_label):
    cnt = focal_df['primary_label'].value_counts().get(primary_label, 0)
    return primary_label if cnt >= CFG['folds'] else '__rare__'

for fold_idx, (tr_idx, va_idx) in enumerate(
    skf.split(focal_df, focal_df['primary_label'].map(_rare_label))
):
    print(f'\nFold {fold_idx + 1}/{CFG["folds"]}')

    focal_tr = focal_df.iloc[tr_idx].reset_index(drop=True)
    focal_va = focal_df.iloc[va_idx].reset_index(drop=True)

    focal_tr_dl = DataLoader(
        FocalDataset(focal_tr, EMBD_DIR, train=True),
        batch_size=CFG['batch_size'], shuffle=True,
        num_workers=CFG['num_workers'], collate_fn=seq_collate,
        drop_last=True, pin_memory=_use_amp)
    focal_va_dl = DataLoader(
        FocalDataset(focal_va, EMBD_DIR, train=False),
        batch_size=CFG['batch_size'], shuffle=False,
        num_workers=CFG['num_workers'], collate_fn=seq_collate,
        drop_last=False, pin_memory=_use_amp)
    sc_tr_dl = DataLoader(
        SoundscapeSeqDataset(seq_groups, EMBD_DIR, train=True),
        batch_size=max(1, CFG['batch_size'] // 4), shuffle=True,
        num_workers=CFG['num_workers'], collate_fn=seq_collate,
        drop_last=True, pin_memory=_use_amp)

    model     = PerchGRU(n_classes, CFG['perch_emb_dim'],
                         CFG['gru_hidden'], CFG['gru_layers'], CFG['gru_dropout']).to(device)
    optimizer = AdamW(model.parameters(), lr=CFG['lr'], weight_decay=1e-4)
    scaler    = GradScaler(enabled=_use_amp)
    warmup    = LinearLR(optimizer, start_factor=0.1, end_factor=1.0, total_iters=CFG['warmup_epochs'])
    cosine    = CosineAnnealingLR(optimizer, T_max=max(1, CFG['epochs'] - CFG['warmup_epochs']), eta_min=1e-6)
    scheduler = SequentialLR(optimizer, schedulers=[warmup, cosine], milestones=[CFG['warmup_epochs']])

    best_auc    = -1.0
    patience_ct = 0
    best_state  = None
    swa_states  = []
    _swa_ep     = max(1, int(CFG['epochs'] * CFG['swa_start_frac']))

    def _compute_loss(logits, y_pad, mask, w):
        loss_per  = _criterion(logits, y_pad)
        m         = mask.unsqueeze(-1).float()
        step_loss = (loss_per * m).sum(dim=[1, 2]) / (m.sum(dim=[1, 2]).clamp(min=1))
        return (step_loss * w).mean()

    for epoch in range(CFG['epochs']):
        model.train()
        train_loss = 0.0
        n_batches  = 0
        sc_iter = iter(sc_tr_dl)
        for xf, yf, mf, wf in focal_tr_dl:
            for src_x, src_y, src_m, src_w in [
                (xf, yf, mf, wf),
                *([next(sc_iter, (None,)*4)])
            ]:
                if src_x is None:
                    continue
                src_x = src_x.to(device); src_y = src_y.to(device)
                src_m = src_m.to(device); src_w = src_w.to(device)
                optimizer.zero_grad()
                with autocast(enabled=_use_amp):
                    logits = model(src_x)
                    loss   = _compute_loss(logits, src_y, src_m, src_w)
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                train_loss += loss.item()
                n_batches  += 1
        train_loss /= max(n_batches, 1)
        scheduler.step()

        model.eval()
        val_preds, val_targets = [], []
        val_loss = 0.0
        with torch.no_grad():
            for xv, yv, mv, wv in focal_va_dl:
                xv, yv = xv.to(device), yv.to(device)
                with autocast(enabled=_use_amp):
                    logits_v = model(xv).float()[:, 0, :]
                yv_sq = yv[:, 0, :]
                val_loss += F.binary_cross_entropy_with_logits(logits_v, yv_sq.to(device)).item()
                val_preds.append(torch.sigmoid(logits_v).cpu().numpy())
                val_targets.append(yv_sq.cpu().numpy())
        val_loss /= max(len(focal_va_dl), 1)

        fp = np.vstack(val_preds)   if val_preds   else np.zeros((len(va_idx), n_classes))
        ft = np.vstack(val_targets) if val_targets else np.zeros((len(va_idx), n_classes))
        ft_bin = (ft >= 0.5).astype(np.float32)
        auc_ep = [
            roc_auc_score(ft_bin[:, j], fp[:, j])
            for j in range(n_classes)
            if ft_bin[:, j].sum() > 0 and (1 - ft_bin[:, j]).sum() > 0
        ]
        val_auc = np.mean(auc_ep) if auc_ep else 0.0

        if val_auc > best_auc:
            best_auc    = val_auc
            patience_ct = 0
            best_state  = copy.deepcopy(model.state_dict())
        else:
            patience_ct += 1

        if epoch >= _swa_ep:
            swa_states.append(copy.deepcopy(model.state_dict()))
            if len(swa_states) > 10: swa_states.pop(0)

        if (epoch + 1) % 5 == 0 or patience_ct == 0:
            print(f'  Ep {epoch+1:3d}: train={train_loss:.4f}  val={val_loss:.4f}  auc={val_auc:.4f}')

        if patience_ct >= CFG['patience']:
            print(f'  Early stop @ epoch {epoch+1}')
            break

    if best_state is None:
        best_state = copy.deepcopy(model.state_dict())
    if swa_states:
        avg_state = {k: torch.stack([s[k].float() for s in swa_states]).mean(0).to(swa_states[0][k].dtype)
                     for k in swa_states[0]}
        model.load_state_dict(avg_state)
        print(f'  SWA: averaged {len(swa_states)} ckpts')
    else:
        model.load_state_dict(best_state)

    ckpt = f'{_out_root}/perch_gru_v25_fold{fold_idx}.pt'
    torch.save(model.state_dict(), ckpt)
    gru_fold_scores.append(best_auc)
    print(f'  Fold {fold_idx+1} best AUC: {best_auc:.4f}  saved {ckpt}')
    del model
    if torch.cuda.is_available(): torch.cuda.empty_cache()

print(f'\n\u2705 GRU Mean OOF AUC: {np.mean(gru_fold_scores):.4f} \u00b1 {np.std(gru_fold_scores):.4f}')
print(f'   Fold AUCs: {[f"{a:.4f}" for a in gru_fold_scores]}')

In [ ]:
# === CELL 9: TRAIN MEL FOLDS (ResNet18 + EfficientNet-B0 on correct species list) ===
print('=' * 65)
print(f'v25 Mel Training   {CFG["folds"]} folds  (ResNet18 + EfficientNet-B0)')
print(f'  Species list from taxonomy.csv — same as GRU')
print(f'  Waveform cache: {WAV_CACHE_DIR}')
print('=' * 65)

mel_df = df.copy()
if 'secondary_labels' not in mel_df.columns:
    mel_df['secondary_labels'] = '[]'
else:
    mel_df['secondary_labels'] = mel_df['secondary_labels'].fillna('[]')
mel_df = mel_df[mel_df['primary_label'].isin(species_set)].reset_index(drop=True)
print(f'Mel training clips: {len(mel_df)}')

mel_fold_scores = {arch: [] for arch in ['resnet18', 'efficientnet_b0']}

for fold_idx, (tr_idx, va_idx) in enumerate(
    skf.split(mel_df, mel_df['primary_label'].map(lambda x: x if mel_df['primary_label'].value_counts().get(x,0) >= CFG['folds'] else '__rare__'))
):
    print(f'\nFold {fold_idx + 1}/{CFG["folds"]}')

    mel_tr = mel_df.iloc[tr_idx].reset_index(drop=True)
    mel_va = mel_df.iloc[va_idx].reset_index(drop=True)

    tr_ds = MelFocalDataset(mel_tr, TRAIN_AUDIO_DIR, train=True,  cache_dir=WAV_CACHE_DIR)
    va_ds = MelFocalDataset(mel_va, TRAIN_AUDIO_DIR, train=False, cache_dir=WAV_CACHE_DIR)
    tr_dl = DataLoader(tr_ds, batch_size=CFG['mel_batch'], shuffle=True,
                       num_workers=CFG['num_workers'], drop_last=True, pin_memory=_use_amp)
    va_dl = DataLoader(va_ds, batch_size=CFG['mel_batch'], shuffle=False,
                       num_workers=CFG['num_workers'], drop_last=False, pin_memory=_use_amp)

    for arch in ['resnet18', 'efficientnet_b0']:
        print(f'  Training {arch}...')
        model     = BirdCLEFModel(arch, n_classes, pretrained=True).to(device)
        optimizer = AdamW(model.parameters(), lr=CFG['mel_lr'], weight_decay=1e-4)
        scaler    = GradScaler(enabled=_use_amp)
        warmup    = LinearLR(optimizer, start_factor=0.1, end_factor=1.0, total_iters=CFG['warmup_epochs'])
        cosine    = CosineAnnealingLR(optimizer, T_max=max(1, CFG['mel_epochs'] - CFG['warmup_epochs']), eta_min=1e-6)
        scheduler = SequentialLR(optimizer, schedulers=[warmup, cosine], milestones=[CFG['warmup_epochs']])

        best_auc    = -1.0
        patience_ct = 0
        best_state  = None

        for epoch in range(CFG['mel_epochs']):
            model.train()
            train_loss = 0.0
            for xb, yb in tr_dl:
                xb, yb = xb.to(device), yb.to(device)
                optimizer.zero_grad()
                with autocast(enabled=_use_amp):
                    loss = F.binary_cross_entropy_with_logits(model(xb), yb)
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                train_loss += loss.item()
            train_loss /= max(len(tr_dl), 1)
            scheduler.step()

            model.eval()
            val_preds, val_targets = [], []
            with torch.no_grad():
                for xv, yv in va_dl:
                    xv = xv.to(device)
                    with autocast(enabled=_use_amp):
                        preds = torch.sigmoid(model(xv).float()).cpu().numpy()
                    val_preds.append(preds)
                    val_targets.append(yv.numpy())

            fp = np.vstack(val_preds)
            ft = np.vstack(val_targets)
            ft_bin = (ft >= 0.5).astype(np.float32)
            auc_ep = [
                roc_auc_score(ft_bin[:, j], fp[:, j])
                for j in range(n_classes)
                if ft_bin[:, j].sum() > 0 and (1 - ft_bin[:, j]).sum() > 0
            ]
            val_auc = np.mean(auc_ep) if auc_ep else 0.0

            if val_auc > best_auc:
                best_auc    = val_auc
                patience_ct = 0
                best_state  = copy.deepcopy(model.state_dict())
            else:
                patience_ct += 1

            if (epoch + 1) % 5 == 0 or patience_ct == 0:
                print(f'    Ep {epoch+1:3d}: train={train_loss:.4f}  auc={val_auc:.4f}')

            if patience_ct >= CFG['mel_patience']:
                print(f'    Early stop @ epoch {epoch+1}')
                break

        if best_state:
            model.load_state_dict(best_state)
        ckpt = f'{_out_root}/{arch}_v25_fold{fold_idx}.pt'
        torch.save(model.state_dict(), ckpt)
        mel_fold_scores[arch].append(best_auc)
        print(f'    {arch} fold {fold_idx+1} AUC: {best_auc:.4f}  saved {ckpt}')
        del model
        if torch.cuda.is_available(): torch.cuda.empty_cache()

for arch, scores in mel_fold_scores.items():
    if scores:
        print(f'\n✅ {arch} Mean OOF AUC: {np.mean(scores):.4f} ± {np.std(scores):.4f}')


In [ ]:
# === CELL 10: SUMMARY ===
saved = sorted(Path(_out_root).glob('*_v25_fold*.pt'))
print('Checkpoints saved to /kaggle/working:')
for f in saved:
    print(f'  {f.name}  ({f.stat().st_size / 1e6:.1f} MB)')

print()
print('GRU fold AUCs :', [f'{a:.4f}' for a in gru_fold_scores])
print('GRU mean AUC  :', f'{np.mean(gru_fold_scores):.4f}')
for arch, scores in mel_fold_scores.items():
    if scores:
        print(f'{arch} mean AUC: {np.mean(scores):.4f}')

print()
print('Next steps:')
print('  1. Output tab -> New Dataset -> name: birdclef-2026-weights-v25')
print('  2. Create inference-v25-ensemble.ipynb (update checkpoint names and GRU arch)')
print('  3. Inference CFG: gru_hidden=768, gru_layers=3')
print('  4. Suggested starting weights: gru=0.6, mel=0.4 (test both are now comparable)')